# Private CayleyPy Results Ingest npm gate
CPU-only dependency, test, and TypeScript verification. No deployment.


In [ ]:
from __future__ import annotations

import base64
import hashlib
import io
import json
import os
import shutil
import subprocess
import tarfile
import urllib.request
import zipfile
from pathlib import Path

WORKING = Path("/kaggle/working")
ROOT = WORKING / "cayleypy-results-ingest-gate"
PACKAGE = ROOT / "services" / "cayleypy-results-ingest"
NPM_CACHE = Path("/tmp/cayleypy-results-ingest-npm-cache")
NODE_ROOT = Path("/tmp/cayleypy-results-ingest-node")
NODE_VERSION = "v22.23.1"
NODE_ARCHIVE_NAME = "node-" + NODE_VERSION + "-linux-x64.tar.xz"
NODE_BASE_URL = "https://nodejs.org/dist/" + NODE_VERSION
PAYLOAD_B64 = '...'
EXPECTED_SHA256 = {'configs/cayleypy_results_schema_v1.json': '5dc4888c70836229e0fb31508cf967082a186e472091d58826450368c2a9f126', 'services/cayleypy-results-ingest/package.json': '80e59683dc86dbeb7d12fb1200da52a1bfd40573c084acee55b41e61fb816929', 'services/cayleypy-results-ingest/package-lock.json': '6c84152419596a537b4f0df8dee1ab2e843103faf606506bd46a6483fcf10b81', 'services/cayleypy-results-ingest/tsconfig.json': '68f0b9d9dcfcb08ab1d2164b5463cbd670ab6f8ad62f8623a7bb7434d35ea8fa', 'services/cayleypy-results-ingest/wrangler.jsonc': '21018636ac49a9836762f03aa28160e53a9b50cbf4e9e850a5579ca7845a770c', 'services/cayleypy-results-ingest/vitest.config.ts': '81f596085ac0467e05bf8b99976aa1ff17475271151d4725d9a0b7c183084e21', 'services/cayleypy-results-ingest/vitest.schema.config.ts': 'e06f4a73c0c2a9a662c2a9e2485e0fda83fcdae435efa0b7207960fa7fdc2d9a', 'services/cayleypy-results-ingest/migrations/0001_initial.sql': '8c3fe6fdc4381e123a901593962194f90aae7bfc484e6a6f5685195d05cb0ba6', 'services/cayleypy-results-ingest/migrations/0002_ingest_rate_limits.sql': '803e8b3af0dc4e1af2ddd32301a14fdba9d543cf9d5512a934f2d581f85e28f4', 'services/cayleypy-results-ingest/src/schema.ts': '5b2d0cdc1736bb5af720624d6f069567975d06991821e6a22776eacc3199126f', 'services/cayleypy-results-ingest/src/ids.ts': 'beec2434b4d4c96eaf7aeef3258f938f2e1ca6c123389096987365ff8e6727a6', 'services/cayleypy-results-ingest/src/db.ts': 'd46481ee7eb8a7d2cbc3673f305566b31b6605eb4c6cae1d7547cfbec72a6609', 'services/cayleypy-results-ingest/src/storage.ts': '929a71468e5f5773e1c6aad8416360a93e46213b5f715819dd8bdd9fc9338c8b', 'services/cayleypy-results-ingest/src/worker.ts': 'f45a1fb11fb80c93a872e20645faa3c7e822075b640347b64f3be8a93259dc11', 'services/cayleypy-results-ingest/test/schema.test.ts': '28ff4675db6dc1be505e9d9591f0491e84dab8d4cf6b7fad24895fab79f4900c', 'services/cayleypy-results-ingest/test/apply-migrations.ts': 'f7cc52a868ed3bf5ae2cf93fead32335a1541019d7828e16eb1375b0ebe5918d', 'services/cayleypy-results-ingest/test/receipt.test.ts': '686b3beb31adb722f1647bbd8a9ab95807edb06419222178cfe436eb1b2e6094', 'services/cayleypy-results-ingest/test/worker.test.ts': '4a3f514a3e33f47599aa529c72e175d1211d3b74962518c0105dee1c89363ddc'}

if ROOT.exists():
    shutil.rmtree(ROOT)
if NPM_CACHE.exists():
    shutil.rmtree(NPM_CACHE)
if NODE_ROOT.exists():
    shutil.rmtree(NODE_ROOT)
ROOT.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD_B64))) as archive:
    archive.extractall(ROOT)

shasums = urllib.request.urlopen(
    NODE_BASE_URL + "/SHASUMS256.txt", timeout=60
).read().decode("utf-8")
expected_node_sha = next(
    line.split()[0]
    for line in shasums.splitlines()
    if line.endswith("  " + NODE_ARCHIVE_NAME)
)
node_archive = urllib.request.urlopen(
    NODE_BASE_URL + "/" + NODE_ARCHIVE_NAME, timeout=180
).read()
observed_node_sha = hashlib.sha256(node_archive).hexdigest()
if observed_node_sha != expected_node_sha:
    raise RuntimeError("Node archive checksum mismatch")
NODE_ROOT.mkdir(parents=True)
with tarfile.open(fileobj=io.BytesIO(node_archive), mode="r:xz") as archive:
    archive.extractall(NODE_ROOT)
NODE_BIN = NODE_ROOT / NODE_ARCHIVE_NAME.removesuffix(".tar.xz") / "bin"
os.environ["PATH"] = str(NODE_BIN) + os.pathsep + os.environ["PATH"]
(WORKING / "node-bootstrap.json").write_text(
    json.dumps(
        {"version": NODE_VERSION, "archive": NODE_ARCHIVE_NAME, "sha256": observed_node_sha},
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)

observed = {}
for relative in EXPECTED_SHA256:
    observed[relative] = hashlib.sha256((ROOT / relative).read_bytes()).hexdigest()
if observed != EXPECTED_SHA256:
    raise RuntimeError("embedded payload checksum mismatch")
(WORKING / "payload-sha256.json").write_text(
    json.dumps(observed, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

combined_log = WORKING / "npm-gate.log"
combined_log.write_text("", encoding="utf-8")
results = {"payload_sha256": observed, "commands": []}
command_outputs = {}
command_stdout = {}
RUN_ENV = {
    **os.environ,
    "NPM_CONFIG_CACHE": str(NPM_CACHE),
    "NPM_CONFIG_REGISTRY": "https://registry.npmjs.org/",
    "NPM_CONFIG_FETCH_RETRIES": "3",
    "NPM_CONFIG_UPDATE_NOTIFIER": "false",
    "NO_COLOR": "1",
    "FORCE_COLOR": "0",
}


def run(label: str, argv: list[str]) -> int:
    completed = subprocess.run(
        argv,
        cwd=PACKAGE,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        env=RUN_ENV,
    )
    command_stdout[label] = completed.stdout
    output = completed.stdout + completed.stderr
    command_outputs[label] = output
    section = f"\n===== {label} (exit={completed.returncode}) =====\n{output}"
    print(section, flush=True)
    with combined_log.open("a", encoding="utf-8") as handle:
        handle.write(section)
    (WORKING / f"{label}.log").write_text(output, encoding="utf-8")
    results["commands"].append(
        {"label": label, "argv": argv, "exit_code": completed.returncode}
    )
    return completed.returncode

REGISTRY_QUERIES = {
    "npm-view-vitest-pool": [
        "npm", "view", "@cloudflare/vitest-pool-workers@0.19.0",
        "version", "dist-tags", "dependencies", "peerDependencies", "engines", "dist.integrity", "--json",
    ],
    "npm-view-wrangler": [
        "npm", "view", "wrangler@4.115.0",
        "version", "dist-tags", "dependencies", "engines", "dist.integrity", "--json",
    ],
    "npm-view-workers-types": [
        "npm", "view", "@cloudflare/workers-types@5.20260729.1",
        "version", "dist-tags", "dependencies", "engines", "dist.integrity", "--json",
    ],
    "npm-view-vitest": [
        "npm", "view", "vitest@4.1.10",
        "version", "dist-tags", "dependencies", "peerDependencies", "engines", "dist.integrity", "--json",
    ],
    "npm-view-workerd": [
        "npm", "view", "workerd@1.20260729.1",
        "version", "dist-tags", "dependencies", "engines", "dist.integrity", "--json",
    ],
}
REGISTRY_EXPECTED = {
    "npm-view-vitest-pool": "0.19.0",
    "npm-view-wrangler": "4.115.0",
    "npm-view-workers-types": "5.20260729.1",
    "npm-view-vitest": "4.1.10",
    "npm-view-workerd": "1.20260729.1",
}

run("node-version", ["node", "--version"])
run("npm-version", ["npm", "--version"])
for label, argv in REGISTRY_QUERIES.items():
    run(label, argv)
run("npm-install-package-lock-only", ["npm", "install", "--package-lock-only", "--no-audit", "--no-fund"])
run("npm-ci", ["npm", "ci", "--no-audit", "--no-fund"])
run(
    "npm-ls-critical",
    [
        "npm", "ls", "@cloudflare/vitest-pool-workers", "@cloudflare/workers-types",
        "vitest", "wrangler", "miniflare", "workerd", "--json",
    ],
)
run("npm-test", ["npm", "test"])
run("npm-test-worker", ["npm", "exec", "--", "vitest", "run", "--config", "vitest.config.ts"])
run("npm-typecheck", ["npm", "run", "typecheck"])

registry_metadata = {}
registry_parse_errors = []
for label, expected_version in REGISTRY_EXPECTED.items():
    try:
        metadata = json.loads(command_stdout[label])
    except (KeyError, json.JSONDecodeError) as exc:
        registry_parse_errors.append({"label": label, "error": type(exc).__name__})
        continue
    registry_metadata[label] = metadata
registry_exact = not registry_parse_errors and all(
    registry_metadata[label].get("version") == expected
    for label, expected in REGISTRY_EXPECTED.items()
)
(WORKING / "registry-metadata.json").write_text(
    json.dumps(
        {
            "expected_versions": REGISTRY_EXPECTED,
            "exact": registry_exact,
            "parse_errors": registry_parse_errors,
            "registry": registry_metadata,
        },
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)

lockfile = PACKAGE / "package-lock.json"
lock = json.loads(lockfile.read_text(encoding="utf-8")) if lockfile.is_file() else {"packages": {}}
lock_packages = lock.get("packages", {})
workerd_paths = sorted(
    path for path in lock_packages if path == "node_modules/workerd" or path.endswith("/node_modules/workerd")
)
workerd_versions = sorted({
    lock_packages[path].get("version") for path in workerd_paths if lock_packages[path].get("version")
})

def installed_version(relative: str) -> str | None:
    path = PACKAGE / "node_modules" / relative / "package.json"
    if not path.is_file():
        return None
    return json.loads(path.read_text(encoding="utf-8"))["version"]

expected_stack = {
    "@cloudflare/vitest-pool-workers": "0.19.0",
    "@cloudflare/workers-types": "5.20260729.1",
    "vitest": "4.1.10",
    "wrangler": "4.115.0",
    "miniflare": "4.20260722.1",
    "workerd": "1.20260729.1",
}
resolved_stack = {name: installed_version(name) for name in expected_stack}
resolved_stack_exact = (
    resolved_stack == expected_stack
    and workerd_versions == ["1.20260729.1"]
    and bool(workerd_paths)
)
resolved_report = {
    "expected": expected_stack,
    "resolved": resolved_stack,
    "exact": resolved_stack_exact,
    "lockfile_workerd_paths": workerd_paths,
    "lockfile_workerd_versions": workerd_versions,
}
(WORKING / "resolved-stack.json").write_text(
    json.dumps(resolved_report, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

warning_markers = (
    "newer than the latest supported date",
    "falling back to",
    "unsupported compatibility date",
    "does not support compatibility date",
)
warning_hits = []
for label, output in command_outputs.items():
    for line_number, line in enumerate(output.splitlines(), start=1):
        lowered = line.lower()
        if any(marker in lowered for marker in warning_markers):
            warning_hits.append({"label": label, "line": line_number, "text": line})
warning_report = {
    "compatibility_date": "2026-07-28",
    "markers": list(warning_markers),
    "hits": warning_hits,
    "passed": not warning_hits,
}
(WORKING / "compatibility-warning-scan.json").write_text(
    json.dumps(warning_report, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

post_install_sha256 = {
    relative: hashlib.sha256((ROOT / relative).read_bytes()).hexdigest()
    for relative in EXPECTED_SHA256
}
post_install_mismatches = {
    relative: {"expected": EXPECTED_SHA256[relative], "observed": observed_sha}
    for relative, observed_sha in post_install_sha256.items()
    if observed_sha != EXPECTED_SHA256[relative]
}
(WORKING / "post-install-payload-sha256.json").write_text(
    json.dumps(post_install_sha256, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

prior_commands_passed = all(item["exit_code"] == 0 for item in results["commands"])
gate_assertions = {
    "prior_commands_passed": prior_commands_passed,
    "registry_versions_exact": registry_exact,
    "resolved_stack_exact": resolved_stack_exact,
    "workerd_override_exact": workerd_versions == ["1.20260729.1"],
    "compatibility_warning_scan_clean": not warning_hits,
    "post_install_payload_exact": not post_install_mismatches,
    "lockfile_present": lockfile.is_file(),
}
gate_exit = 0 if all(gate_assertions.values()) else 1
gate_output = json.dumps(gate_assertions, indent=2, sort_keys=True) + "\n"
command_outputs["gate-assertions"] = gate_output
(WORKING / "gate-assertions.log").write_text(gate_output, encoding="utf-8")
with combined_log.open("a", encoding="utf-8") as handle:
    handle.write(f"\n===== gate-assertions (exit={gate_exit}) =====\n{gate_output}")
results["commands"].append(
    {"label": "gate-assertions", "argv": [], "exit_code": gate_exit}
)

if lockfile.is_file():
    shutil.copy2(lockfile, WORKING / "package-lock.json")
results.update(
    {
        "lockfile_present": lockfile.is_file(),
        "registry_versions_exact": registry_exact,
        "resolved_stack": resolved_report,
        "compatibility_warning_scan": warning_report,
        "post_install_payload_sha256": post_install_sha256,
        "post_install_payload_mismatches": post_install_mismatches,
        "gate_assertions": gate_assertions,
        "all_commands_passed": all(item["exit_code"] == 0 for item in results["commands"]),
    }
)
(WORKING / "npm-gate-results.json").write_text(
    json.dumps(results, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
shutil.rmtree(ROOT)
shutil.rmtree(NPM_CACHE)
shutil.rmtree(NODE_ROOT)
print(json.dumps(results, indent=2, sort_keys=True), flush=True)
